# Vectors and Similarity

## Overview
In this notebook, I will use basic vector math to demonstrate cosine similarity when comparing multiple text documents.  I will create `add_vec()`, `subtract_vec()`, `scale_vec()`, `dot_product_vec()`, `length_vec()`, and finally `cosine_similarity()` functions to handle the computations.  I'll also leverage the use of a simple tokenizer to split texts from a few separate books into vocabulary that will be used to determine cosine similarity between a user query and the texts in question.  

This will emulate at a very fundamental level tokenizing, embedding, and ranking.   

### Why? 
I am doing this to gain a fundamental understanding of the mathematics involved to solve basic Machine Learning tasks.  I eventually want to take this knowledge and create my own classifier or regressor model without relying on custom libraries.  

### Constraints
I will not use any special libraries to achieve this, just simple Python.  My goal is to learn how to do vector math without needing any of it.  

### Steps

1.  Create mathematical functions needed to perform cosine similarity
2.  Create a vectorizer for the texts that will be indexed
3.  Query against texts and rank them by cosine similarity

## Create Functions
We begin with basic vector addition. A `vector` is simply a measurement of the displacement in a certain direction, expressed by an ordered list of entries.  Vectors exist within a dimension $\mathbb{R^{n}}$ and can only be added to other vectors with the same dimension.  Vectors can be added in any order because they have properties of commutativity.    

*Why do we care so much about vectors?*  Because they allow us to express raw data in numbers.  We can encode that raw data into numbers and directions to be used for constructing complex relationships between pieces of data.  It's a central theme to Machine Learning.    

We can express all vectors as such: $\textbf{v}, \textbf{w} \in \mathbb{R^{n}}$.  (Vector V and Vector W exist within a set of dimension $\mathbb{R}^{n}$)

### Vector Addition, Subtraction, and Scaling 

We can define vector addition as:
$$
\textbf{v} + \textbf{w} = (v_1 + w_1, v_2 + w_2,...,v_n + w_n)
$$

Before we add, one important point needs to be reiterated:  Python expresses dimensionality as length, and like stated earlier all vectors need to be equal in dimension to be added or subtracted from one another.  

We will first make a function to compare dimensionality/length of the vectors.  This will be reusable for other functions we make:

In [1]:
# Compare dimensionality of 2 vectors
def compare_dim_vec(v, w):
    if len(v) != len(w):
        raise ValueError('Vectors must be of equal length')

Now we will create a function that can add 2 vectors together:

In [2]:
# Create 2 vectors for re-use throughout exercise
vec1 = [1, 2]
vec2 = [3, 4]

def add_vec(v, w):
    compare_dim_vec(v, w)
    return [v[i] + w[i] for i in range(len(v))]

print(add_vec(vec1, vec2)) # Expect [4, 6]
        

[4, 6]


Vector subtraction is very similar and also works in a predictable way. However, unlike addition it is *not* commutative.  Order of subtraction does matter.  

We can define vector subtraction as:
$$
\textbf{v} - \textbf{w} = (v_1 - w_1, v_2 - w_2,...,v_n - w_n)
$$

Let's create that in Python:

In [3]:
def subtract_vec(v, w):
    compare_dim_vec(v, w)
    return [v[i] - w[i] for i in range(len(v))]

print(subtract_vec(vec1, vec2)) # Expect [-2, -2]

[-2, -2]


Vectors have 2 components: length, and direction.  That is the displacement we mentioned earlier.  
    
One way to modify length without changing direction (only reversing it) is through the use of **scalar multiplication**.  We can "scale" or stretch a vector by $c$ (a constant).

This is expressed like so:
$$
cv = (cv_1, cv_2,...,cv_n)
$$

If $c = 0$, then the vector is a zero vector that cannot be divided. 
If $c = -1$, then we have $-\textbf{v}$.  

Let's create a Python function to scale one of our vectors:

In [4]:
def scale_vec(c, v):
    return [c * i for i in v]

print(scale_vec(5, vec1)) # Expect [5, 10]

[5, 10]


We can also apply this to a **linear combination**, where we can apply that constant across the addition of 2 vectors:

$$c_1v_1 + c_1w_1$$

In [5]:
# Example linear combination where c = 5
# 5[1, 2] + 5[3, 4]
# [5, 10] + [15, 20]
# [20, 30]
print(add_vec((scale_vec(5, vec1)), (scale_vec(5, vec2))))

[20, 30]


### Dot Product

The **Dot Product** provides an answer for how to navigate the higher dimensionality of vectors.  When vectors exceed the 2D and 3D that we are familiar with, the dot product provides us a way of measuring through those dimensions because it returns to us only a **scalar** (single number) value.  

The **Dot Product** is strictly defined as the result of an operation between two vectors.  Assuming $\textbf{v} = (v_1, v_2,...,v_n) \in \mathbb{R^{n}}$ and $\textbf{w} - (w_1, w_2,...w_n) \in \mathbb{R^{n}}$, then:
$$
\textbf{v} \cdot \textbf{w} = (v_1w_1 + v_2w_2 +... + v_nw_n)
$$

From a geometric perspective, the dot product is a way of measuring overlap between $v$ and $w$.  As we rotate $w$ away from $v$, the dot product trends towards 0.  

Let's write it now in Python:

In [6]:
def dot_product_vec(v, w):
    compare_dim_vec(v, w)
    product = 0
    for i in range(len(v)):
        product += v[i] * w[i]
    return product

print(dot_product_vec(vec1, vec2))
    

11


### Vector Length (Norm/Magnitude)

As stated earlier, in higher dimensions we want to be able to measure the length of a vector even if that becomes increasingly less understandable in dimensional space greater than 3D.  

By taking the square root of the dot product we computed previously, we can derive the "length" of the vector (not the Python definition), also called the **norm**, the **L2 norm**, or often called **magnitude**.  It is also known as **Euclidean normalization**.  

This produces a scalar value.  

We express vector length as:
$$
\lVert{v}\rVert = \sqrt{\textbf{v} \cdot \textbf{v}}
$$

In [7]:
def length_vec(v):
    return dot_product_vec(v, v)**0.5
print(length_vec(vec1))

2.23606797749979


Not to be confused with the **norm** that we previously calculated, it is also possible to *normalize* a vector by simply dividing $v$ by its length.  This is useful when direction matters more than length:
$$
\textbf{u} = \frac{\textbf{v}}{\lVert\textbf{v}\rVert}
$$

In [8]:
def normalize_vec(v):
    length = length_vec(v)
    if length == 0:
        raise ValueError('Cannot divide zero vector')
    return [x / length for x in v]

print(normalize_vec(vec1)) # Expect [0.45, 0.89]

[0.4472135954999579, 0.8944271909999159]


### Cosine Similarity

If we assume $\textbf{v}, \textbf{w} \in \mathbb{R^{n}}$, then at standard position (0,0) or the **origin** we can use $v$, $w$, and $v - w$ as 3 sides of a triangle.  

Applying the Law of Cosines to this (https://en.wikipedia.org/wiki/Law_of_cosines), we can determine the angle between $v$ and $w$:
$$
\lVert\textbf{v} - \textbf{w}\rVert^2 = \lVert\textbf{v}\rVert^2 + \lVert\textbf{w}\rVert^2 - 2\lVert{v}\rVert\lVert{w}\rVert\cos\theta
$$

Therefore:
$$
\cos\theta = \frac{\textbf{v}\cdot\textbf{w}}{\lVert{v}\rVert\lVert{w}\rVert}
$$

**Cosine Similarity** is a way of expressing similarity via direction while normalizing magnitude.  It doesn't care as much about length as it does direction, which makes it extremely useful for Machine Learning tasks like text similarity where proportionality is often preferred to quantity.  

For example, if I was a user who wanted to learn about feeding cats and I matched with a text with only the word "cats" on it, that would clearly meet my requirement to find something about cats by definition, but would not provide anything meaningful about feeding them.  Thats' where **cosine similarity** is preferred. 

Let's code cosine similarity in Python.  Fortunately this makes use of a lot of the math we've already done:

In [9]:
def cosine_similarity(v, w):
    dot = dot_product_vec(v, w)
    denom = length_vec(v) * length_vec(w)
    if denom == 0:
        return 0.0 # Catch all to say it's a zero vector
    return dot / denom

print(cosine_similarity(vec1, vec2)) #Expect >0.98, they're very similar vectors

0.9838699100999074


## Create a Vectorizer

In order to glue these concepts together, we'll create a simple comparison tool based on cosine similarity.  

We will:

- Establish a vocabulary (chosen words we will base comparisons on)
- Choose and split texts (classic literature)
- Vectorize them all
- Practice user queries and check cosine similarity between the user query and the texts

### Establish a Vocabulary

We will now pick a vocabulary to index against.  It will be used as the basis of comparison.  What's important is the order and the words included, as they will be used as the comparison to all the other texts included:

In [10]:
vocabulary = ['sea', 'boat', 'ocean', 'sailor', 'fishing', 'fish', 'whales', 'swells', 'storm', 'storms', 'waves', 'scary', 'fear', 'rain', 'deep', 'wide', 'lonely', 'daring', 'trying', 'struggling', 'awe', 'rocks', 'beach', 'dock', 'vessel']
vocabulary = sorted(vocabulary)
print(vocabulary)

['awe', 'beach', 'boat', 'daring', 'deep', 'dock', 'fear', 'fish', 'fishing', 'lonely', 'ocean', 'rain', 'rocks', 'sailor', 'scary', 'sea', 'storm', 'storms', 'struggling', 'swells', 'trying', 'vessel', 'waves', 'whales', 'wide']


### Create Splitter

Next we will create a simple splitter that can break down long texts into digestible chunks.  Our goal here isn't to create anything complicated, just to find a way to break text into individual words that can be used to create a vocabulary we can reference when computing cosine similarity.  Effectively, each text will be turned into a vector.  When a user makes a query containing any of the vocabulary included in those texts, the similarity will be calculated.  

In order to do that, we need to break the texts down.  

I've selected a few texts to be tokenized from famous literature: 

In [11]:
old_man_and_the_sea = '''He no longer dreamed of storms, nor of women, nor of great occurrences, nor of great fish, nor fights, nor contests of strength, nor of his wife. He only dreamed of places now and the lions on the beach. They played like young cats in the dusk and he loved them as he loved the boy. He never dreamed about the boy. He simply woke, looked out the open door at the moon and unrolled his trousers and put them on.'''

the_count_of_monte_cristo = '''Life is a storm, my young friend. You will bask in the sunlight one moment, be shattered on the rocks the next. What makes you a man is what you do when that storm comes. You must look into that storm and shout as you did in Rome. Do your worst, for I will do mine! Then the fates will know you as we know you'''

heart_of_darkness = '''It seems to me I am trying to tell you a dream--making a vain attempt, because no relation of a dream can convey the dream-sensation, that commingling of absurdity, surprise, and bewilderment in a tremor of struggling revolt, that notion of being captured by the incredible which is of the very essence of dreams...No, it is impossible; it is impossible to convey the life-sensation of any given epoch of one's existence--that which makes its truth, its meaning--its subtle and penetrating essence. It is impossible. We live, as we dream-alone...'''

In [12]:
print(old_man_and_the_sea[0:25])
print(the_count_of_monte_cristo[0:25])
print(heart_of_darkness[0:25])

He no longer dreamed of s
Life is a storm, my young
It seems to me I am tryin


In [13]:
# Create a dictionary to store the texts in k,v format
dict_of_texts = {
    'old_man_and_the_sea': old_man_and_the_sea, 
    'the_count_of_monte_cristo': the_count_of_monte_cristo, 
    'heart_of_darkness': heart_of_darkness
}
print(dict_of_texts.keys())

dict_keys(['old_man_and_the_sea', 'the_count_of_monte_cristo', 'heart_of_darkness'])


### Splitting Text

First we need to split the text up.  This emulates at a very basic level what a tokenizer does.  We can do this using the `split()` method in Python.  We will use a regex statement to filter out characters and aid with the splitting:

In [14]:
import re

def splitter(dict_of_texts):
    '''
    Performs splitting and stripping of text
    '''
    split_text_dict = {}
    for k, v in dict_of_texts.items():
        split_text = re.split(r'([,.:;?_!"()\']|--|\s)', v) 
        split_text = [text.strip() for text in split_text if text.strip()] # Get rid of whitespace completely
        split_text_dict[k] = sorted(split_text)
    return split_text_dict

split_text_dict = splitter(dict_of_texts)

print('Keys:', split_text_dict.keys(), '\n')
print('Values:', split_text_dict.values())

Keys: dict_keys(['old_man_and_the_sea', 'the_count_of_monte_cristo', 'heart_of_darkness']) 

Values: dict_values([[',', ',', ',', ',', ',', ',', ',', '.', '.', '.', '.', '.', 'He', 'He', 'He', 'He', 'They', 'about', 'and', 'and', 'and', 'and', 'as', 'at', 'beach', 'boy', 'boy', 'cats', 'contests', 'door', 'dreamed', 'dreamed', 'dreamed', 'dusk', 'fights', 'fish', 'great', 'great', 'he', 'he', 'his', 'his', 'in', 'like', 'lions', 'longer', 'looked', 'loved', 'loved', 'moon', 'never', 'no', 'nor', 'nor', 'nor', 'nor', 'nor', 'nor', 'now', 'occurrences', 'of', 'of', 'of', 'of', 'of', 'of', 'of', 'on', 'on', 'only', 'open', 'out', 'places', 'played', 'put', 'simply', 'storms', 'strength', 'the', 'the', 'the', 'the', 'the', 'the', 'the', 'them', 'them', 'trousers', 'unrolled', 'wife', 'woke', 'women', 'young'], ['!', ',', ',', ',', '.', '.', '.', '.', 'Do', 'I', 'Life', 'Rome', 'Then', 'What', 'You', 'You', 'a', 'a', 'and', 'as', 'as', 'bask', 'be', 'comes', 'did', 'do', 'do', 'fates', 'for

### Vectorizing Text

Our next step is to make each text into its own vector.  We can think of this as a *very* shallow form of an embedding model.  We do this by creating vectors associated to the texts stored in `split_test_dict`, referencing the chosen vocabulary as the comparison point.  

First we make a `vectorize()` function that takes a text_dict and the vocabulary, then uses that to count similar words.  

In [15]:
# Vectorize function
def vectorize(text_dict, vocabulary):
    '''
    Creates vectors associated to indeces provided by vocabulary list,
    creates counts based on those indeces
    '''
    word_to_index = {word: index for index, word in enumerate(vocabulary)}
    vectors = {}
    for name, tokens in text_dict.items():
        vec = [0] * len(vocabulary)
        for token in tokens:
            if token in word_to_index:
                vec[word_to_index[token]] += 1
        vectors[name] = vec
    return vectors

We will want to take the texts we split up earlier and vectorize them now using this new function:

In [16]:
text_vectors = vectorize(split_text_dict, vocabulary)
print(text_vectors)

{'old_man_and_the_sea': [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], 'the_count_of_monte_cristo': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0], 'heart_of_darkness': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0]}


Above you'll see that we've vectorized the texts according to the `vocabulary` we set earlier.  While they are very sparse embeddings, they will suffice.  That vocabulary defined the indexes that we're referencing against, and also the dimensionality of the resulting vector.  As each text is reviewed and matched against words appearing in `vocabulary`, we increment if there's a common match and it happens multiple times.

We can vectorize a user query the same way we do texts.  It will still be measured against the same `vocabulary` and then indexed appropriately.  

Let's imitate a user query now using the same methodology we've used so far:

In [17]:
user_query_dict = {
    'query': "Find me a book about fish, storms, and women"
}

split_user_query = splitter(user_query_dict)

print('Keys:', split_user_query.keys(), '\n')
print('Values:', split_user_query.values())

Keys: dict_keys(['query']) 

Values: dict_values([[',', ',', 'Find', 'a', 'about', 'and', 'book', 'fish', 'me', 'storms', 'women']])


In [18]:
query_vector = vectorize(split_user_query, vocabulary)['query']
print(query_vector)

[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]


## Query Against Texts and Rank by Similarity

We've reached the point now where we have what is necessary to rank similarity of the user query to the provided texts.  We will do that by simply ranking the documents based on both dot product and cosine similarity.  

Why dot product as well?  Each one of them gets a different result.  

- We might use dot product if we care more about the number of matches vs. the proportionality of the matches themselves.  Based on the above query, we'd see a much higher dot product score with any text that explicitly mentioned 'sailing', 'sea', 'struggle', and 'loneliness' many times.  

- Cosine similarity cares more about the proportion of similar vectors.  Seeing more matching items even if they only appear once or twice is more important than the quantity of hits.  

With that said, let's do the comparison:

In [19]:
# Rank by Dot Product
def rank_by_dot(query_vec, text_vec):
    '''
    Creates an empty list of scores and uses dot product to score, 
    then sorts the docs by highest dot product
    '''
    scores = []
    for name, vec in text_vec.items():
        score = dot_product_vec(query_vec, vec)
        scores.append((name, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

print('Dot Product Rank:', rank_by_dot(query_vector, text_vectors))

# Rank by Cosine Similarity
def rank_by_cos(query_vec, text_vec):
    '''
    Creates an empty list of scores and uses cosine similarity to score, 
    then sorts the docs by highest similarity
    '''
    scores = [] # We'll create a scoring list 
    for name, vec in text_vec.items():
        score = cosine_similarity(query_vec, vec)
        scores.append((name, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores
print('Cosine Similarity Rank:', rank_by_cos(query_vector, text_vectors))

Dot Product Rank: [('old_man_and_the_sea', 2), ('the_count_of_monte_cristo', 0), ('heart_of_darkness', 0)]
Cosine Similarity Rank: [('old_man_and_the_sea', 0.8164965809277259), ('the_count_of_monte_cristo', 0.0), ('heart_of_darkness', 0.0)]


Based on our original query, which was: "Find me a book about fish, storms, and women", we've found that 'The Old Man and the Sea' had the most similarity by dot product and cosine similarity.  There were more explicity mentions of certain words, and there were overall more word mentions too.  

Let's try with a slightly different query:

In [20]:
# New Query 
user_query_dict = { 'query': 'Find me a book about a fish, a storm, and a woman' }
split_user_query = splitter(user_query_dict)
query_vector = vectorize(split_user_query, vocabulary)['query']
print(query_vector)

# New Results
print('Dot Product Rank:', rank_by_dot(query_vector, text_vectors))
print('Cosine Similarity Rank:', rank_by_cos(query_vector, text_vectors))

[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Dot Product Rank: [('the_count_of_monte_cristo', 3), ('old_man_and_the_sea', 1), ('heart_of_darkness', 0)]
Cosine Similarity Rank: [('the_count_of_monte_cristo', 0.6708203932499369), ('old_man_and_the_sea', 0.40824829046386296), ('heart_of_darkness', 0.0)]


Interestingly enough, in spite of the topic being almost exactly the same, the similarity has shifted considerably.  By simply using a different word (storm instead of storms), we've aligned more directionally with the 'Count of Monte Cristo' due to the fact that passage explicitly mentioned the word storm multiple times, and more words overall matched.  'Heart of Darkness' doesn't match at all.  

This is one of the biggest limitations of the 'bag of words' approach. It relies on an exact match of a word to gain similarity vs. semantic understanding of the topic, which might include variations of the word and theme.  

Also, the vocabulary that we chose limits us to matches.  Because of the nautical words we picked, naturally passages that include any mention of those will have greater overlap.  That's the main reason why 'Heart of Darkness' isn't going to show similarity often here: it only had 2 matches with the vocabulary at all.  

## Conclusion

We've done basic vector math, developed a way of comparing vectors, actually compared vectorized texts and queries, and seen the results.  

While this similarity tool wasn't particularly useful, it did show us the underlying mathematics of what complex systems do all the time: taking raw data, converting them into numbers in the form of vectors, and then use them as a way of creating relationships.  Underlying Retrieval Augmented Generation (RAG) systems is something quite similar.  They leverage a vector database which takes unstructured data, stores it as vectors, and then draws relationships between that data.  Queries are then compared to those stored vectors.  Although the method is far more comprehensive for RAG, at a base level they are very similar. 

From here we could upgrade this to something more capable by integrating a real embedding model or varying the chunking strategies, as well as a few other items.  

### References

- I used 3 classic literature textbooks as sources for the quotes used in the text:  The Old Man and the Sea by Ernest Hemingway, The Count of Monte Cristo by Alexandre Dumas, and Heart of Darkness by Joseph Conrad.  
- I primarily used the book "Linear and Matrix Algebra" by Nathaniel Johnston to learn the vector conventions seen in this notebook.  

Conrad, J. (1899). *Heart of darkness*. Blackwood's Magazine.

Dumas, A. (1844). *The count of Monte Cristo*. Pétion.

Hemingway, E. (1952). *The old man and the sea*. Charles Scribner's Sons.

Johnston, N. (2021). *Introduction to linear and matrix algebra*. Springer.
https://njohnston.ca/publications/introduction-to-linear-and-matrix-algebra/